# Datasets:

## Library Installs and Imports

In [1]:
%pip install requests numpy pandas matplotlib
import requests, numpy as np, pandas as pd


Note: you may need to restart the kernel to use updated packages.


## Collecting Data 

### Biking Data 

In [2]:
###get biking data 
bridge_url = 'https://data.seattle.gov/api/v3/views/65db-xm6k/query.json'
r = requests.get(bridge_url).json()

In [3]:
###initially process data 
###drop irrelevant information, convert time to datetime object
bike_df = pd.DataFrame(r).drop(columns=[':id',':version',':updated_at',':created_at'])

###converting date to timezone aware 
is_dst = ~bike_df['date'].duplicated(keep='first')
bike_df.date = pd.to_datetime(bike_df.date, format='%Y-%m-%dT%H:%M:%S.%f').dt.tz_localize('America/Los_Angeles', ambiguous=is_dst, nonexistent='NaT')

#convert strings to floats 
num_cols = bike_df.select_dtypes(str).columns.to_list()
bike_df[num_cols] = bike_df[num_cols].apply(lambda x: pd.to_numeric(x), axis=0)

###dropping nan dates
bike_df = bike_df.dropna(subset='date')

####dropping nan bike counts, will eventually impute data by local average
bike_df = bike_df.dropna()

In [4]:
print('10 highest Days:\n_____________________')
bike_df.sort_values('fremont_bridge', ascending=False)[:10]


10 highest Days:
_____________________


,date,fremont_bridge,fremont_bridge_nb,fremont_bridge_sb
63318,2019-06-11 17:00:00-07:00,1097.0,850.0,247.0
61470,2019-08-27 17:00:00-07:00,1071.0,826.0,245.0
62982,2019-06-25 17:00:00-07:00,1052.0,795.0,257.0
63798,2019-05-22 17:00:00-07:00,1004.0,745.0,259.0
61806,2019-08-13 17:00:00-07:00,998.0,780.0,218.0
61278,2019-09-04 17:00:00-07:00,997.0,790.0,207.0
63486,2019-06-04 17:00:00-07:00,995.0,742.0,253.0
63342,2019-06-10 17:00:00-07:00,993.0,765.0,228.0
61782,2019-08-14 17:00:00-07:00,989.0,737.0,252.0
61110,2019-09-11 17:00:00-07:00,983.0,768.0,215.0


In [5]:
####save data to csv 
bike_df.to_csv('scraped_data/fremont_bridge_counter.csv', index=False)

### Weather Data

In [1]:
###set up weather data api 

import os, requests, pandas as pd, time, json 

API_key = os.getenv('weather-api')
lat, lon = [47.61191234063561, -122.33851982527909]
units = 'imperial'

####get historical weather data for period the fremont bridge counter is active 
from datetime import datetime
###set start date to unix version of fremont bridge csv
start = bike_df.date.min().timestamp().__int__()
end = bike_df.date.max().timestamp().__int__()


print('We will need to make ', np.round((end-start)/3600/20).__int__(), ' calls total.')

NameError: name 'bike_df' is not defined

In [ ]:
prev_records = pd.read_csv('scraped_data/weather.csv')

,dt,temp,feels_like,pressure,humidity,dew_point,uvi,clouds,visibility,wind_speed,wind_deg,wind_gust,date,weather_main,weather_desc,rain_1hr_mm,snow_1hr_mm
0,1349182800,52.93,51.35,1022,73,44.53,0,31,10000,11.50,180,18.39,2012-10-02 06:00:00-07:00,Clouds,scattered clouds,0.0,0.0
1,1349186400,53.04,51.71,1022,78,46.38,0,29,10000,10.36,180,12.64,2012-10-02 07:00:00-07:00,Clouds,scattered clouds,0.0,0.0
2,1349190000,54.52,52.63,1022,63,42.21,0,30,10000,11.50,180,19.55,2012-10-02 08:00:00-07:00,Clouds,scattered clouds,0.0,0.0
3,1349193600,55.63,53.44,1022,54,39.27,0,29,10000,6.89,180,10.36,2012-10-02 09:00:00-07:00,Clouds,scattered clouds,0.0,0.0
4,1349197200,57.25,55.26,1022,55,41.25,0,43,10000,4.61,180,8.05,2012-10-02 10:00:00-07:00,Clouds,scattered clouds,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121957,1788228000,68.76,68.43,1008,66,56.95,0,90,10000,5.99,17,8.99,2026-08-31 19:00:00-07:00,Clouds,overcast clouds,0.0,0.0
121958,1788231600,66.33,65.80,1008,67,55.06,0,72,10000,7.00,338,8.01,2026-08-31 20:00:00-07:00,Clouds,broken clouds,0.0,0.0
121959,1788235200,64.56,63.82,1008,66,52.97,0,12,10000,3.27,6,NaN,2026-08-31 21:00:00-07:00,Clouds,few clouds,0.0,0.0
121960,1788238800,63.72,63.09,1009,70,53.76,0,7,10000,1.01,285,1.99,2026-08-31 22:00:00-07:00,Clear,clear sky,0.0,0.0


In [7]:
###loop to get data from start time to end time, not the most efficient but it works for now 
weather_data = pd.DataFrame()
CHECKPOINT = 'scraped_data/weather_checkpoint.pkl'

###check if there is a more recent checkpoint to start off from 
if os.path.exists(CHECKPOINT):
    records = pd.read_pickle(CHECKPOINT).to_dict('records')
    start = (max(rec['dt'] for rec in records) + 1 ).__int__()
    print(f'Resuming from {datetime.fromtimestamp(start)} - {len(records)} hours of data already saved.')
else:
    records = []
    start = bike_df.date.min().timestamp().__int__()

###initialization
weather_url = f'https://api.openweathermap.org/data/4.0/onecall/timeline/1h?lat={lat}&lon={lon}&units={units}&start={start}&appid={API_key}'
calls = 0
max_calls = 1200
session = requests.Session()

try:
    while weather_url and start < end:

        print('Data is within range. Requesting...')
        resp = session.get(weather_url, timeout=30)
        resp.raise_for_status()
        r = resp.json()

    ##if no response, break 
        batch = r.get('data', [])
        if not batch:   
            break
        
        records.extend(batch)
        calls += 1

        ###ends when date end is reached 
        if batch[-1]['dt'] >= end:
            break 

        ###save every 20 calls just in case 
        if calls % 20 == 0:
            pd.DataFrame(records).to_pickle(CHECKPOINT)
        data_time = datetime.fromtimestamp(r['data'][0]['dt'])

        ###break at 1000 calls - free daily limit  
        if calls == max_calls:
            break

        print(f'Data Saved for {data_time} - Call: {calls}')

        weather_url = r.get('next')
        time.sleep(1)
except requests.HTTPError:
    print('\nReached max calls for the day! Try again tomorrow.\n')
finally:
    pd.DataFrame(records).to_pickle(CHECKPOINT)
    print(f'{len(records)} hourly records saved. ')

    

Resuming from 2025-06-10 09:00:01 - 111220 hours of data already saved.
Data is within range. Requesting...
Data Saved for 2025-06-10 10:00:00 - Call: 1
Data is within range. Requesting...
Data Saved for 2025-06-11 06:00:00 - Call: 2
Data is within range. Requesting...
Data Saved for 2025-06-12 02:00:00 - Call: 3
Data is within range. Requesting...
Data Saved for 2025-06-12 22:00:00 - Call: 4
Data is within range. Requesting...
Data Saved for 2025-06-13 18:00:00 - Call: 5
Data is within range. Requesting...
Data Saved for 2025-06-14 14:00:00 - Call: 6
Data is within range. Requesting...
Data Saved for 2025-06-15 10:00:00 - Call: 7
Data is within range. Requesting...
Data Saved for 2025-06-16 06:00:00 - Call: 8
Data is within range. Requesting...
Data Saved for 2025-06-17 02:00:00 - Call: 9
Data is within range. Requesting...
Data Saved for 2025-06-17 22:00:00 - Call: 10
Data is within range. Requesting...
Data Saved for 2025-06-18 18:00:00 - Call: 11
Data is within range. Requesting...

In [8]:
weather_df = pd.DataFrame(records).drop_duplicates('dt').sort_values('dt').reset_index(drop=True)
weather_df = weather_df[weather_df['dt'] <= end]
weather_df['date'] = (pd.to_datetime(weather_df['dt'], unit='s', utc=True).dt.tz_convert('America/Los_Angeles'))

###FLATTEN weather data 
w = weather_df['weather'].str[0]
weather_df['weather_main'] = w.str['main']
weather_df['weather_desc'] = w.str['description']

for col in ['rain', 'snow']:
    if col in weather_df:
        weather_df[f'{col}_1hr_mm'] = weather_df[col].str['1h'].fillna(0)

weather_df = weather_df.drop(columns=[c for c in ['weather','rain','snow'] if c in weather_df])
weather_df.to_csv('scraped_data/weather.csv', index=False)


### Fuel Pricing Data

### Merging Bike and Weather Data, Visualizations

In [9]:
####Merge datasets to prepare to train model. 

bike_df
weather_df = pd.read_csv('scraped_data/weather.csv')

print('weather_df dates as a percent of bike_df dates:', '\nData Collected: {}%'.format(np.round(weather_df.shape[0]/bike_df.shape[0]*100,2)))

###set date column to datetime
weather_df.date = pd.to_datetime(weather_df.date, utc=True).dt.tz_convert('America/Los_Angeles')

df_merged = pd.merge(bike_df, weather_df, on='date')

print('Total observations in common: {}'.format(df_merged.shape[0]))

weather_df dates as a percent of bike_df dates: 
Data Collected: 100.03%
Total observations in common: 121925
